### Project 6 - Advanced Solution

We are going to build the push pipeline in small steps.

The final data flow will look like this:

```text
cars.csv
   ↓
CSV parser and sniffer
   ↓
normalization coroutine
   ↓
validation coroutine
   ↓
filter coroutine
   ↓
CSV saving coroutine
```

Although the project only requires filtering the vehicle name, we will keep each stage generic. That gives us a pipeline that can be reused for other CSV files and predicates.

Let's begin with the imports we are going to need:

In [10]:
import csv
from contextlib import contextmanager
from functools import wraps
from pathlib import Path

Before parsing the file, we need to determine how it is structured.

The `csv.Sniffer` class can usually detect whether the file uses commas, semicolons, tabs, or pipes. Since sniffing is heuristic, we will also add a fallback that looks for the delimiter used most consistently across the sample.

In [11]:
def _fallback_delimiter(sample, delimiters=",;\t|"):
    """Choose the most consistently used delimiter in a CSV sample."""
    lines = [
        line
        for line in sample.splitlines()
        if line.strip()
    ][:20]

    if not lines:
        return ","

    def score(delimiter):
        counts = [line.count(delimiter) for line in lines]
        positive_counts = [count for count in counts if count > 0]

        if not positive_counts:
            return (-1, -1, -1)

        coverage = len(positive_counts)
        consistency = -(max(positive_counts) - min(positive_counts))
        frequency = sum(positive_counts)

        return (coverage, consistency, frequency)

    return max(delimiters, key=score)


def _dialect_for_delimiter(delimiter):
    """Create a standard CSV dialect using the specified delimiter."""
    class CustomDialect(csv.excel):
        pass

    CustomDialect.delimiter = delimiter
    return CustomDialect

Now we can write a function that inspects the input file.

This function:

* reads only a sample of the file
* detects the delimiter
* detects whether the first row looks like a header
* reads the possible header row
* returns all of that information in a dictionary

An explicit delimiter can still be supplied when we do not want automatic detection.

In [12]:
def inspect_csv(
    f_name,
    input_delimiter=None,
    detect_header=True,
    delimiters=",;\t|",
    sample_size=16384,
    encoding="utf-8-sig",
):
    """Inspect a CSV file and return its dialect and header information."""
    path = Path(f_name)

    if not path.is_file():
        raise FileNotFoundError("CSV file not found: {}".format(path))

    if input_delimiter is not None:
        if not isinstance(input_delimiter, str) or len(input_delimiter) != 1:
            raise ValueError(
                "input_delimiter must be a single character or None"
            )

    with path.open("r", newline="", encoding=encoding) as f:
        sample = f.read(sample_size)

        if input_delimiter is not None:
            dialect = _dialect_for_delimiter(input_delimiter)
            delimiter_source = "explicit"
        elif not sample:
            dialect = csv.excel
            delimiter_source = "default"
        else:
            try:
                dialect = csv.Sniffer().sniff(
                    sample,
                    delimiters=delimiters,
                )
                delimiter_source = "sniffer"
            except csv.Error:
                delimiter = _fallback_delimiter(
                    sample,
                    delimiters=delimiters,
                )
                dialect = _dialect_for_delimiter(delimiter)
                delimiter_source = "fallback"

        header_detected = False

        if detect_header and sample:
            try:
                header_detected = csv.Sniffer().has_header(sample)
            except csv.Error:
                header_detected = False

        f.seek(0)
        reader = csv.reader(
            f,
            dialect=dialect,
            skipinitialspace=True,
        )
        first_row = next(reader, None)

    if first_row:
        first_row = [
            value.strip().lstrip("\ufeff")
            for value in first_row
        ]

    return {
        "dialect": dialect,
        "delimiter": dialect.delimiter,
        "quotechar": dialect.quotechar,
        "delimiter_source": delimiter_source,
        "header_detected": header_detected,
        "first_row": first_row,
    }

Let's add a parser next.

Unlike the original pull-pipeline parser, this version receives the format information we just detected. It also supports three header modes:

* `True` - always skip the first row
* `False` - never skip the first row
* `'auto'` - use the result from `csv.Sniffer().has_header(...)`

The parser itself remains a generator because rows should be read lazily rather than loading the entire file into memory.

In [13]:
def parse_data(
    f_name,
    format_info,
    skip_header="auto",
    stats=None,
    encoding="utf-8-sig",
):
    """Yield parsed CSV rows lazily."""
    if skip_header not in (True, False, "auto"):
        raise ValueError(
            "skip_header must be True, False, or 'auto'"
        )

    should_skip_header = (
        format_info["header_detected"]
        if skip_header == "auto"
        else skip_header
    )

    with Path(f_name).open(
        "r",
        newline="",
        encoding=encoding,
    ) as f:
        reader = csv.reader(
            f,
            dialect=format_info["dialect"],
            skipinitialspace=True,
        )

        if should_skip_header:
            header = next(reader, None)

            if stats is not None:
                stats["header_skipped"] = header is not None
                stats["input_header"] = (
                    [value.strip().lstrip("\ufeff") for value in header]
                    if header
                    else None
                )

        for row in reader:
            if stats is not None:
                stats["rows_read"] += 1

            yield row

Since we are going to create several coroutines, an auto-priming decorator will be useful.

Without this decorator, every coroutine would have to be manually advanced with `next(...)` before its first call to `.send(...)`.

In [14]:
def coroutine(fn):
    """Automatically advance a coroutine to its first yield."""
    @wraps(fn)
    def inner(*args, **kwargs):
        coro = fn(*args, **kwargs)
        next(coro)
        return coro

    return inner

Before writing the pipeline stages, let's create a statistics dictionary.

All stages will share this dictionary. That lets us see how many rows were read, skipped, matched, and written.

In [15]:
def create_stats():
    return {
        "rows_read": 0,
        "rows_normalized": 0,
        "rows_skipped_blank": 0,
        "rows_skipped_invalid": 0,
        "rows_evaluated": 0,
        "rows_matched": 0,
        "rows_written": 0,
        "header_skipped": False,
        "input_header": None,
        "input_delimiter": None,
        "delimiter_source": None,
        "header_detected": False,
    }

We are now ready to write the final stage of the pipeline: the CSV saver.

This coroutine owns the output file. When the coroutine is closed, the `with` block closes and flushes the file automatically.

The output delimiter is configurable, and we can optionally write a header.

In [16]:
@coroutine
def save_csv(
    f_name,
    stats,
    delimiter=",",
    include_header=False,
    header=None,
    encoding="utf-8",
):
    if not isinstance(delimiter, str) or len(delimiter) != 1:
        raise ValueError(
            "The output delimiter must be one character"
        )

    if include_header and header is None:
        raise ValueError(
            "A header is required when include_header=True"
        )

    path = Path(f_name)
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open(
        "w",
        newline="",
        encoding=encoding,
    ) as f:
        writer = csv.writer(
            f,
            delimiter=delimiter,
            lineterminator="\n",
        )

        if include_header:
            writer.writerow(header)

        try:
            while True:
                row = yield
                writer.writerow(row)
                stats["rows_written"] += 1
        except GeneratorExit:
            pass

The first processing stage will normalize rows.

CSV files often contain spaces around fields or a UTF-8 byte-order mark at the beginning of the first value. This coroutine removes those artifacts and can discard blank rows.

In [17]:
@coroutine
def normalize_rows(
    target,
    stats,
    strip_fields=True,
    skip_blank_rows=True,
):
    try:
        while True:
            row = yield

            if strip_fields:
                row = [value.strip() for value in row]

            if row:
                row[0] = row[0].lstrip("\ufeff")

            if skip_blank_rows and not any(row):
                stats["rows_skipped_blank"] += 1
                continue

            stats["rows_normalized"] += 1
            target.send(row)
    except GeneratorExit:
        target.close()

The next stage validates the number of columns.

This is especially helpful when a delimiter has been detected incorrectly: instead of silently treating the entire line as one field, the pipeline can raise a clear error.

The invalid-row policy can be:

* `'raise'` - stop immediately with an error
* `'skip'` - ignore the malformed row and continue

In [18]:
@coroutine
def validate_rows(
    target,
    stats,
    expected_columns=None,
    invalid_row_policy="raise",
):
    if expected_columns is not None and expected_columns <= 0:
        raise ValueError(
            "expected_columns must be positive or None"
        )

    if invalid_row_policy not in ("raise", "skip"):
        raise ValueError(
            "invalid_row_policy must be 'raise' or 'skip'"
        )

    try:
        while True:
            row = yield

            if (
                expected_columns is not None
                and len(row) != expected_columns
            ):
                stats["rows_skipped_invalid"] += 1

                message = (
                    "Expected {} columns but received {}. "
                    "Row: {!r}"
                ).format(
                    expected_columns,
                    len(row),
                    row,
                )

                if invalid_row_policy == "raise":
                    raise ValueError(message)

                continue

            target.send(row)
    except GeneratorExit:
        target.close()

Now we can write the generic filtering coroutine.

It does not know anything about vehicle names. It simply receives a predicate and forwards rows for which that predicate returns `True`.

In [19]:
@coroutine
def filter_rows(filter_pred, target, stats):
    if not callable(filter_pred):
        raise TypeError(
            "filter_pred must be callable"
        )

    try:
        while True:
            row = yield
            stats["rows_evaluated"] += 1

            if filter_pred(row):
                stats["rows_matched"] += 1
                target.send(row)
    except GeneratorExit:
        target.close()

Let's create a few reusable predicate helpers.

`contains_text` checks one field. The `all_of`, `any_of`, and `negate` helpers let us combine predicates without changing the pipeline itself.

In [20]:
def contains_text(
    fragment,
    field_index=0,
    case_sensitive=False,
):
    if not isinstance(fragment, str):
        raise TypeError(
            "fragment must be a string"
        )

    if fragment == "":
        raise ValueError(
            "fragment must not be empty"
        )

    if field_index < 0:
        raise ValueError(
            "field_index must be non-negative"
        )

    needle = (
        fragment
        if case_sensitive
        else fragment.casefold()
    )

    def predicate(row):
        if field_index >= len(row):
            return False

        value = row[field_index]
        haystack = (
            value
            if case_sensitive
            else value.casefold()
        )

        return needle in haystack

    return predicate


def all_of(*predicates):
    return lambda row: all(
        predicate(row)
        for predicate in predicates
    )


def any_of(*predicates):
    return lambda row: any(
        predicate(row)
        for predicate in predicates
    )


def negate(predicate):
    return lambda row: not predicate(row)

The project asks us to accept an arbitrary number of filters on the vehicle name.

The helper below creates one predicate from those fragments:

* `match_mode='all'` requires every fragment
* `match_mode='any'` requires at least one fragment

An empty collection of fragments matches every row.

In [21]:
def build_name_predicate(
    name_filters,
    field_index=0,
    match_mode="all",
    case_sensitive=False,
):
    if match_mode not in ("all", "any"):
        raise ValueError(
            "match_mode must be 'all' or 'any'"
        )

    name_filters = tuple(name_filters)

    if not name_filters:
        return lambda row: True

    predicates = tuple(
        contains_text(
            fragment,
            field_index=field_index,
            case_sensitive=case_sensitive,
        )
        for fragment in name_filters
    )

    if match_mode == "all":
        return all_of(*predicates)

    return any_of(*predicates)

Now we can assemble the stages into a pipeline coroutine.

Pipeline stages are built from the destination backwards:

1. create the CSV saver
2. place the filter in front of it
3. place validation in front of the filter
4. place normalization at the front

Closing the outer pipeline closes every downstream stage.

In [22]:
@coroutine
def pipeline_coro(
    out_file,
    predicate,
    stats,
    expected_columns=None,
    invalid_row_policy="raise",
    strip_fields=True,
    skip_blank_rows=True,
    output_delimiter=",",
    include_header=False,
    output_header=None,
    output_encoding="utf-8",
):
    save = save_csv(
        out_file,
        stats=stats,
        delimiter=output_delimiter,
        include_header=include_header,
        header=output_header,
        encoding=output_encoding,
    )

    filtered = filter_rows(
        predicate,
        target=save,
        stats=stats,
    )

    validated = validate_rows(
        target=filtered,
        stats=stats,
        expected_columns=expected_columns,
        invalid_row_policy=invalid_row_policy,
    )

    normalized = normalize_rows(
        target=validated,
        stats=stats,
        strip_fields=strip_fields,
        skip_blank_rows=skip_blank_rows,
    )

    try:
        while True:
            row = yield
            normalized.send(row)
    except GeneratorExit:
        normalized.close()

As in the broadcasting examples, a context manager is useful for resource management.

The caller only has to use a `with` statement. The pipeline is closed automatically even if an exception occurs while rows are being processed.

In [23]:
@contextmanager
def pipeline(
    out_file,
    predicate,
    stats,
    expected_columns=None,
    invalid_row_policy="raise",
    strip_fields=True,
    skip_blank_rows=True,
    output_delimiter=",",
    include_header=False,
    output_header=None,
    output_encoding="utf-8",
):
    p = pipeline_coro(
        out_file=out_file,
        predicate=predicate,
        stats=stats,
        expected_columns=expected_columns,
        invalid_row_policy=invalid_row_policy,
        strip_fields=strip_fields,
        skip_blank_rows=skip_blank_rows,
        output_delimiter=output_delimiter,
        include_header=include_header,
        output_header=output_header,
        output_encoding=output_encoding,
    )

    try:
        yield p
    finally:
        p.close()

We now have all the individual pieces, so let's create a high-level runner.

This function will:

* inspect the input CSV
* decide whether to skip the header
* optionally infer the expected column count from that header
* build the predicate
* create the coroutine pipeline
* push every parsed row into the pipeline
* return the processing statistics

The source and output paths are also checked to prevent accidentally overwriting the input file.

In [24]:
def run_name_filter_pipeline(
    source_file,
    output_file,
    name_filters,
    name_field_index=0,
    match_mode="all",
    case_sensitive=False,
    input_delimiter=None,
    output_delimiter=",",
    skip_header="auto",
    detect_header=True,
    expected_columns="auto",
    invalid_row_policy="raise",
    strip_fields=True,
    skip_blank_rows=True,
    include_header=False,
    output_header=None,
    input_encoding="utf-8-sig",
    output_encoding="utf-8",
):
    source_path = Path(source_file)
    output_path = Path(output_file)

    if not source_path.is_file():
        raise FileNotFoundError(
            "Source CSV not found: {}".format(source_path)
        )

    if source_path.resolve() == output_path.resolve():
        raise ValueError(
            "The source and output files must be different"
        )

    if skip_header not in (True, False, "auto"):
        raise ValueError(
            "skip_header must be True, False, or 'auto'"
        )

    format_info = inspect_csv(
        source_path,
        input_delimiter=input_delimiter,
        detect_header=detect_header,
        encoding=input_encoding,
    )

    should_skip_header = (
        format_info["header_detected"]
        if skip_header == "auto"
        else skip_header
    )

    detected_header = (
        format_info["first_row"]
        if should_skip_header
        else None
    )

    if expected_columns == "auto":
        expected_columns = (
            len(detected_header)
            if detected_header
            else None
        )
    elif (
        expected_columns is not None
        and not isinstance(expected_columns, int)
    ):
        raise TypeError(
            "expected_columns must be an integer, None, or 'auto'"
        )

    if include_header and output_header is None:
        if detected_header is None:
            raise ValueError(
                "No input header is available to write"
            )

        output_header = detected_header

    predicate = build_name_predicate(
        name_filters=name_filters,
        field_index=name_field_index,
        match_mode=match_mode,
        case_sensitive=case_sensitive,
    )

    stats = create_stats()
    stats.update({
        "source": str(source_path),
        "output": str(output_path),
        "filters": tuple(name_filters),
        "match_mode": match_mode,
        "input_delimiter": format_info["delimiter"],
        "delimiter_source": format_info["delimiter_source"],
        "header_detected": format_info["header_detected"],
        "expected_columns": expected_columns,
    })

    with pipeline(
        out_file=output_path,
        predicate=predicate,
        stats=stats,
        expected_columns=expected_columns,
        invalid_row_policy=invalid_row_policy,
        strip_fields=strip_fields,
        skip_blank_rows=skip_blank_rows,
        output_delimiter=output_delimiter,
        include_header=include_header,
        output_header=output_header,
        output_encoding=output_encoding,
    ) as p:
        for row in parse_data(
            source_path,
            format_info=format_info,
            skip_header=skip_header,
            stats=stats,
            encoding=input_encoding,
        ):
            p.send(row)

    return stats

A small reporting function will make the statistics easier to read:

In [25]:
def print_pipeline_report(stats):
    print("CSV pipeline report")
    print("-------------------")
    print(
        "Input delimiter : {!r} ({})".format(
            stats["input_delimiter"],
            stats["delimiter_source"],
        )
    )
    print(
        "Header detected : {}".format(
            stats["header_detected"]
        )
    )
    print(
        "Header skipped  : {}".format(
            stats["header_skipped"]
        )
    )
    print(
        "Expected columns: {}".format(
            stats["expected_columns"]
        )
    )
    print(
        "Rows read       : {}".format(
            stats["rows_read"]
        )
    )
    print(
        "Rows normalized : {}".format(
            stats["rows_normalized"]
        )
    )
    print(
        "Rows evaluated  : {}".format(
            stats["rows_evaluated"]
        )
    )
    print(
        "Rows matched    : {}".format(
            stats["rows_matched"]
        )
    )
    print(
        "Rows written    : {}".format(
            stats["rows_written"]
        )
    )
    print(
        "Blank skipped   : {}".format(
            stats["rows_skipped_blank"]
        )
    )
    print(
        "Invalid skipped : {}".format(
            stats["rows_skipped_invalid"]
        )
    )

And now we can run the actual project pipeline.

Notice that we do not specify the input delimiter. The sniffer will detect the semicolon used by the supplied `cars.csv` file.

The three name fragments are combined with `match_mode='all'`, so a row must contain all three fragments in the name field.

In [26]:
stats = run_name_filter_pipeline(
    source_file="cars.csv",
    output_file="out.csv",
    name_filters=(
        "Chevrolet",
        "Carlo",
        "Landau",
    ),
    match_mode="all",
    input_delimiter=None,
    output_delimiter=",",
    skip_header="auto",
    expected_columns="auto",
    invalid_row_policy="raise",
)

print_pipeline_report(stats)

CSV pipeline report
-------------------
Input delimiter : ';' (sniffer)
Header detected : True
Header skipped  : True
Expected columns: 9
Rows read       : 406
Rows normalized : 406
Rows evaluated  : 406
Rows matched    : 2
Rows written    : 2
Blank skipped   : 0
Invalid skipped : 0


Finally, let's verify the output file.

Using `csv.reader` for the verification is safer than comparing raw lines because it confirms that each output row really contains nine CSV fields.

In [27]:
with open(
    "out.csv",
    "r",
    newline="",
    encoding="utf-8",
) as f:
    output_rows = list(csv.reader(f))

expected_rows = [
    [
        "Chevrolet Monte Carlo Landau",
        "15.5",
        "8",
        "350.0",
        "170.0",
        "4165.",
        "11.4",
        "77",
        "US",
    ],
    [
        "Chevrolet Monte Carlo Landau",
        "19.2",
        "8",
        "305.0",
        "145.0",
        "3425.",
        "13.2",
        "78",
        "US",
    ],
]

assert output_rows == expected_rows

with open("out.csv", encoding="utf-8") as f:
    for row in f:
        print(row, end="")

Chevrolet Monte Carlo Landau,15.5,8,350.0,170.0,4165.,11.4,77,US
Chevrolet Monte Carlo Landau,19.2,8,305.0,145.0,3425.,13.2,78,US


Perfect!

The pipeline detected the input format, skipped the header, normalized and validated each row, applied an arbitrary number of name filters, and saved the two matching records as comma-separated CSV data.

### A few useful variations

Because the pipeline is generic, we can change its behavior without rewriting any coroutine.

To match either `Ford` or `Chevrolet`, use `match_mode='any'`:

```python
run_name_filter_pipeline(
    'cars.csv',
    'ford_or_chevrolet.csv',
    ('Ford', 'Chevrolet'),
    match_mode='any',
)
```

To copy every valid data row, use an empty filter tuple:

```python
run_name_filter_pipeline(
    'cars.csv',
    'all_cars.csv',
    (),
)
```

To preserve the input header in the output file, set `include_header=True`:

```python
run_name_filter_pipeline(
    'cars.csv',
    'with_header.csv',
    ('Chevrolet',),
    include_header=True,
)
```

To skip malformed rows instead of raising an exception, use:

```python
invalid_row_policy='skip'
```

### Testing the reusable pieces

The following smoke test creates temporary CSV files, so it does not depend on the project dataset.

It checks:

* semicolon delimiter detection
* comma delimiter detection
* automatic header skipping
* `all` matching
* `any` matching
* standard comma-separated output

In [28]:
def run_smoke_tests():
    from tempfile import TemporaryDirectory

    with TemporaryDirectory() as directory:
        directory = Path(directory)

        semicolon_source = directory / "cars_semicolon.csv"
        semicolon_output = directory / "landau.csv"

        semicolon_source.write_text(
            "name;mpg;cylinders\n"
            "Chevrolet Monte Carlo Landau;15.5;8\n"
            "Chevrolet Monte Carlo Landau;19.2;8\n"
            "Ford Mustang;18.0;8\n",
            encoding="utf-8",
        )

        semicolon_stats = run_name_filter_pipeline(
            source_file=semicolon_source,
            output_file=semicolon_output,
            name_filters=(
                "Chevrolet",
                "Carlo",
                "Landau",
            ),
            match_mode="all",
            expected_columns="auto",
        )

        with semicolon_output.open(
            "r",
            newline="",
            encoding="utf-8",
        ) as f:
            semicolon_rows = list(csv.reader(f))

        assert semicolon_stats["input_delimiter"] == ";"
        assert semicolon_stats["rows_written"] == 2
        assert len(semicolon_rows) == 2
        assert all(len(row) == 3 for row in semicolon_rows)

        comma_source = directory / "cars_comma.csv"
        comma_output = directory / "brands.csv"

        comma_source.write_text(
            "name,mpg,cylinders\n"
            "Ford Mustang,18.0,8\n"
            "Chevrolet Nova,22.0,6\n"
            "Toyota Corolla,31.0,4\n",
            encoding="utf-8",
        )

        comma_stats = run_name_filter_pipeline(
            source_file=comma_source,
            output_file=comma_output,
            name_filters=(
                "Ford",
                "Chevrolet",
            ),
            match_mode="any",
            expected_columns="auto",
        )

        with comma_output.open(
            "r",
            newline="",
            encoding="utf-8",
        ) as f:
            comma_rows = list(csv.reader(f))

        assert comma_stats["input_delimiter"] == ","
        assert comma_stats["header_skipped"] is True
        assert [
            row[0]
            for row in comma_rows
        ] == [
            "Ford Mustang",
            "Chevrolet Nova",
        ]

    return "All smoke tests passed."


run_smoke_tests()

'All smoke tests passed.'